In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
import glob
import alto_processing as ap
import json
from tqdm.auto import tqdm
from constants import BASE_PATH

In [4]:

paths = glob.glob(f'{BASE_PATH}**/*.xml', recursive=True)

In [7]:
json_paths  = glob.glob(f'{BASE_PATH}**/processed.json', recursive=True)

In [ ]:
for path in tqdm(paths):
    if path.replace('ocr.xml', 'processed.json') in json_paths:
        continue
    p = path.replace(BASE_PATH, '').split('/')
    lccn = p[0]
    date = f"{p[1]}-{p[2]}-{p[3]}"
    year = p[1]
    month = p[2]
    day = p[3]
    edition = p[4]
    sequence = p[5]
    r = {
        'lccn': lccn,
        'date': date,
        'year': year,
        'month': month,
        'day': day,
        'edition': edition,
        'sequence': sequence,
        'link': f"https://chroniclingamerica.loc.gov/lccn/{lccn}/{date}/{edition}/{sequence}/",
        'text_blocks_raw': ap.extract_text_blocks_from_path(path)
    }
    with open(path.replace('ocr.xml', 'processed.json'), 'w') as json_file:
        json.dump(r, json_file)

# Maybe Old?

In [22]:
path

'data/loc/2013201074/1855/03/03/ed-1/seq-1/ocr.xml'

In [16]:
json_paths  = glob.glob(f'{BASE_PATH}**/processed.json', recursive=True)
processed_json_paths  = [f.replace('_gpt', '') for f in glob.glob(f'{BASE_PATH}**/processed_gpt.json', recursive=True)]

In [17]:
sampled_json_paths = np.random.choice(json_paths, 20)
sampled_json_paths

array(['data/loc/2013201074/1862/06/28/ed-1/seq-4/processed.json',
       'data/loc/2013201074/1867/01/29/ed-1/seq-2/processed.json',
       'data/loc/2013201074/1855/05/03/ed-1/seq-2/processed.json',
       'data/loc/2013201074/1861/05/23/ed-1/seq-4/processed.json',
       'data/loc/2013201074/1862/04/29/ed-1/seq-2/processed.json',
       'data/loc/2013201074/1867/10/01/ed-1/seq-4/processed.json',
       'data/loc/2013201074/1855/10/04/ed-1/seq-4/processed.json',
       'data/loc/2013201074/1860/09/25/ed-1/seq-1/processed.json',
       'data/loc/2013201074/1863/01/03/ed-1/seq-1/processed.json',
       'data/loc/2013201074/1859/04/19/ed-1/seq-2/processed.json',
       'data/loc/2013201074/1862/04/05/ed-1/seq-4/processed.json',
       'data/loc/2013201074/1864/11/10/ed-1/seq-1/processed.json',
       'data/loc/2013201074/1853/01/08/ed-1/seq-2/processed.json',
       'data/loc/2013201074/1864/06/16/ed-1/seq-1/processed.json',
       'data/loc/2013201074/1869/02/02/ed-1/seq-4/processed.js

In [18]:
processed_json_paths

['data/loc/2013201074/1855/11/03/ed-1/seq-2/processed.json',
 'data/loc/2013201074/1855/06/21/ed-1/seq-2/processed.json',
 'data/loc/2013201074/1863/11/24/ed-1/seq-4/processed.json',
 'data/loc/2013201074/1863/11/24/ed-1/seq-2/processed.json',
 'data/loc/2013201074/1865/05/13/ed-1/seq-1/processed.json',
 'data/loc/2013201074/1865/06/13/ed-1/seq-4/processed.json',
 'data/loc/2013201074/1853/04/12/ed-1/seq-1/processed.json',
 'data/loc/2013201074/1853/04/12/ed-1/seq-3/processed.json',
 'data/loc/2013201074/1853/08/13/ed-1/seq-1/processed.json',
 'data/loc/2013201074/1853/06/21/ed-1/seq-2/processed.json',
 'data/loc/2013201074/1853/12/10/ed-1/seq-2/processed.json',
 'data/loc/2013201074/1860/06/09/ed-1/seq-4/processed.json',
 'data/loc/2013201074/1867/02/19/ed-1/seq-4/processed.json',
 'data/loc/2013201074/1867/01/12/ed-1/seq-4/processed.json',
 'data/loc/2013201074/1867/06/20/ed-1/seq-4/processed.json',
 'data/loc/2013201074/1866/03/15/ed-1/seq-4/processed.json',
 'data/loc/2013201074/18

In [19]:
SYSTEM_PROMPT = """You are an expert Spanish OCR post-correction engine for historical newspapers.

TASK
You will receive a JSON array of objects. Each object has:
- "raw_text": a string containing OCR'ed Spanish text (often with punctuation noise, diacritics errors, broken words, and line breaks).

Return a JSON array of objects of the same length and order.
Each output object MUST contain exactly:
- "raw_text": the original raw_text unchanged
- "gpt_fixed_conservative": a conservative OCR-corrected version
- "gpt_fixed_best_guess": a best-guess OCR-corrected version using broader context

--------------------------------
GENERAL RULES (apply to BOTH outputs)
--------------------------------
- Preserve the original meaning and language (Spanish).
- Preserve formatting:
  - Keep line breaks (\n) exactly where they appear.
  - Preserve paragraph structure.
- Do NOT translate, summarize, or modernize language.
- Do NOT add new content that is not plausibly present in the original.
- Output MUST be valid JSON only (no markdown, no explanations).

--------------------------------
RULES FOR `gpt_fixed_conservative` (CONSERVATIVE)
--------------------------------
1) Only correct text when you have HIGH confidence it is an OCR error.
2) If a word or fragment is ambiguous, leave it unchanged.
3) Fix common OCR artifacts when confidence is high:
   - Character confusions: l/I/1, O/0, rn/m, c/e, a/o, u/v, ñ/n, cl/d, h/b, s/f, t/l
   - Obvious broken words where the intended word is clear
   - Clear diacritic errors (e.g., "mas" → "más" only when clearly required)
4) Do NOT guess missing words or restructure sentences.

--------------------------------
RULES FOR `gpt_fixed_best_guess` (BEST GUESS)
--------------------------------
1) Use broader sentence- and paragraph-level context to infer likely words.
2) You MAY:
   - Reconstruct heavily corrupted words if a strong contextual match exists
   - Resolve broken words across spaces or line breaks
   - Restore plausible punctuation and diacritics
3) Prefer historically plausible vocabulary (19th–early 20th century Spanish).
4) If multiple reconstructions are possible and none is clearly better, leave the text unchanged.
5) Do NOT hallucinate:
   - Do not invent facts, names, or technical terms not strongly implied by context.
   - If confidence is low, fall back to the original text.

--------------------------------
STRICT OUTPUT CONSTRAINTS
--------------------------------
- Output MUST be a single JSON array.
- Each array element MUST be an object with exactly:
  "raw_text", "gpt_fixed_conservative", "gpt_fixed_best_guess"
- "raw_text" MUST match the input exactly (byte-for-byte).
- Preserve all line breaks exactly in both processed fields.

--------------------------------
FORMAT EXAMPLE (structure only)
--------------------------------
Input:
[
  {"raw_text":"manmesto la injusticia\\n"},
  {"raw_text":"dictaron la sentencia contra man\\n"}
]

Output:
[
  {
    "raw_text":"manmesto la injusticia\\n",
    "gpt_fixed_conservative":"manifiesto la injusticia\\n",
    "gpt_fixed_best_guess":"manifiesto la injusticia\\n"
  },
  {
    "raw_text":"dictaron la sentencia contra man\\n",
    "gpt_fixed_conservative":"dictaron la sentencia contra man\\n",
    "gpt_fixed_best_guess":"dictaron la sentencia contra Juan\\n"
  }
]
"""

TEMPLATE = """
Process the following JSON array according to the rules.
Return the corrected output JSON only.

INPUT:
{input_json}
"""

In [10]:
from openai import OpenAI
client = OpenAI()

In [11]:
sampled_json_paths

array(['data/loc/2013201074/1855/11/03/ed-1/seq-2/processed.json',
       'data/loc/2013201074/1866/01/13/ed-1/seq-3/processed.json',
       'data/loc/2013201074/1863/11/24/ed-1/seq-4/processed.json',
       'data/loc/2013201074/1866/07/12/ed-1/seq-3/processed.json',
       'data/loc/2013201074/1866/01/27/ed-1/seq-3/processed.json',
       'data/loc/2013201074/1869/12/21/ed-1/seq-3/processed.json',
       'data/loc/2013201074/1857/10/17/ed-1/seq-4/processed.json',
       'data/loc/2013201074/1863/09/24/ed-1/seq-1/processed.json',
       'data/loc/2013201074/1861/07/06/ed-1/seq-2/processed.json',
       'data/loc/2013201074/1868/04/28/ed-1/seq-3/processed.json',
       'data/loc/2013201074/1853/04/07/ed-1/seq-4/processed.json',
       'data/loc/2013201074/1867/12/28/ed-1/seq-4/processed.json',
       'data/loc/2013201074/1855/01/11/ed-1/seq-1/processed.json',
       'data/loc/2013201074/1865/11/28/ed-1/seq-2/processed.json',
       'data/loc/2013201074/1862/10/09/ed-1/seq-3/processed.js

In [ ]:
for path in sampled_json_paths:
    print(path)
    if path in processed_json_paths:
        print("Processed", path)
        continue
    
    with open(path, 'r') as f:
        d = json.load(f)
        
    input_txt = json.dumps([{'raw_text': t['text']} for t in d['text_blocks_raw']])
    response = client.responses.create(
        model="gpt-5-mini",
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": TEMPLATE.format(input_json=input_txt)},
        ],
    )
    
    d['gpt_fixed'] = json.loads(repair_json(response.output_text))
    with open(path.replace('processed.json', 'processed_gpt.json'), 'w') as json_file:
        json.dump(d, json_file)

In [29]:
json_paths  = glob.glob(f'{BASE_PATH}**/processed_gpt.json', recursive=True)

In [34]:
for path in json_paths:
    print(path)
    with open(path, 'r') as f:
        d = json.load(f)
        break

data/loc/2013201074/1855/06/21/ed-1/seq-2/processed_gpt.json


In [39]:
t

{'raw_text': "las mipas del Nohe.;; r r,:nv w'vai A ütJ.tolü\n.oíi? spSundatUmÍDa pfrece; una; escena tqoo jQ va)",
 'gpt_fixed_conservative': "las mipas del Nohe.;; r r,:nv w'vai A ütJ.tolü\n.oíi? spSundatUmÍDa pfrece; una; escena tqoo jQ va)",
 'gpt_fixed_best_guess': 'en las minas del Norte.\nLa segunda lámina ofrece una escena que...'}

In [41]:
for t in d['gpt_fixed']:
    print("raw_text:", t['raw_text'])
    print('\n\n')
    # print("gpt_fixed_best_guess:", t['gpt_fixed_best_guess'])
    print("gpt_fixed_conservative:", t['gpt_fixed_conservative'])
    print('\n\n')
    print('~' * 100)

raw_text: casos análogos puedan ocurrir en esa provincia. Y de la propia Real órden comunicada
por el Sr. Ministro do la Guerra, lo traslado
á V. E. para su conocimiento y efectos correspondientes." ,;( ,rf vi ::u K : ' i
De órden de S. E. se inserta en la Gaceta
del Gobierno. Puerto-Rico 18 de Juntó de 1855.
El Teniente Coronel Comandantet Gefe de
E. M. interino. Paulino García.



gpt_fixed_conservative: casos análogos puedan ocurrir en esa provincia. Y de la propia Real órden comunicada
por el Sr. Ministro de la Guerra, lo traslado
á V. E. para su conocimiento y efectos correspondientes." ,;( ,rf vi ::u K : ' i
De orden de S. E. se inserta en la Gaceta
del Gobierno. Puerto-Rico 18 de Junio de 1855.
El Teniente Coronel Comandante Gefe de
E. M. interino. Paulino García.



~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
raw_text: El Excmo: Sr.k Subsecretario de la Guer-
4 ra con'fecha 19 de Jlbril último dice al Excelentísimo Sr Cap